## Execution Mode
`pytorchsim_functional_mode` runs the kernels on Spike (the outputs are real);
`pytorchsim_timing_mode` times them on TOGSim. Both are keys in the machine file. The timing mode
is fixed when a process imports torch, so each mode below runs in its own process.

In [ ]:
!grep -H -E '^(num_cores|pytorchsim_|codegen_)' togsim_configs/togsim_config.yml togsim_configs/togsim_config_functional_only.yml togsim_configs/togsim_config_timing_only.yml

In [ ]:
%%writefile matmul.py
import torch

device = torch.device("npu:0")

input = torch.randn(512, 512, dtype=torch.bfloat16).to(device=device)
weight = torch.randn(512, 512, dtype=torch.bfloat16).to(device=device)

opt_fn = torch.compile(dynamic=False)(torch.matmul)
npu_output = opt_fn(input, weight)

In [ ]:
import glob
import os
os.environ["TORCHSIM_LOG_PATH"] = os.path.join(os.getcwd(), "togsim_results")
_seen = set(glob.glob(os.path.join(os.environ["TORCHSIM_LOG_PATH"], "*.log")))

def new_logs():
    """The TOGSim logs written since the last call, oldest first (a resident mm writes two:
    its operand's copy-in, then its body; their cycles add up)."""
    logs = set(glob.glob(os.path.join(os.environ["TORCHSIM_LOG_PATH"], "*.log"))) - _seen
    _seen.update(logs)
    return sorted(logs, key=os.path.getmtime)

### Functional & Timing mode

In [ ]:
!TORCHSIM_TARGET=togsim_configs/togsim_config.yml TORCHSIM_DUMP_PATH=$PWD/outputs/functional_timing python matmul.py

In [ ]:
for log in new_logs():
    !grep "Total execution cycles" {log}

### Functional-only mode
No TOGSim: no cycles are reported.

In [ ]:
!TORCHSIM_TARGET=togsim_configs/togsim_config_functional_only.yml TORCHSIM_DUMP_PATH=$PWD/outputs/functional_only python matmul.py

### Timing-only mode
No Spike: the output tensors keep whatever they held, only the cycles are computed.

In [ ]:
!TORCHSIM_TARGET=togsim_configs/togsim_config_timing_only.yml TORCHSIM_DUMP_PATH=$PWD/outputs/timing_only python matmul.py

In [ ]:
for log in new_logs():
    !grep "Total execution cycles" {log}